# OpenHPCの起動

---

OpenHPC環境を構成するVCノード、VCディスクを作成します。

## 構成

このNotebookが構築する環境の構成を図に示します。

![構成](images/ohpc-000.png)

## 準備

このNotebookを実行するための準備作業を行います。

### UnitGroup名の指定

構築環境の UnitGroup名を指定します。

> 「010-パラメータ設定.ipynb」と同じUnitGroup名(`ugroup_name`)を指定することで、既に入力したパラメータを引き継ぐことができます。

VCノードを作成時に指定した値を確認するために `group_vars` ファイル名の一覧を表示します。

In [ ]:
!ls -1 group_vars/*.yml | sed -e 's/^group_vars\///' -e 's/\.yml//' | sort

UnitGroup名を次のセルに指定してください。

In [ ]:
# (例)
# ugroup_name = 'OpenHPC'

ugroup_name = 

### VCCアクセストークンの入力

VCノードを起動するにはVC Controller(VCC)にアクセスして、操作を行う必要があります。VCCにアクセスするために必要となるアクセストークンをここで入力します。

In [ ]:
from getpass import getpass
vcc_access_token = getpass()

入力されたアクセストークンが正しいことを、実際にVCCにアクセスして確認します。

In [ ]:
from common import logsetting
from vcpsdk.vcpsdk import VcpSDK
vcp = VcpSDK(vcc_access_token)

上のセルの実行結果がエラーとなり以下のようなメッセージが表示されている場合は、入力されたアクセストークンに誤りがあります。

```
config vc failed: http_status(403)
2021/XX/XX XX:XX:XX UTC: VCPAuthException: xxxxxxx:token lookup is failed: permission denied
```

エラーになった場合はこの節のセルを全て `unfreeze` してから、もう一度アクセストークンの入力を行ってください。

> `unfreeze`するにはNotebookのツールバーにある`unfreeze below in section`ボタンなどを利用してください。

#### VCCのバージョンチェック

このテンプレートはVCC 25.04以上での利用を想定しています。利用環境のVCCが条件を満たしていることをチェックします。

VCCのバージョンを確認します。

In [ ]:
 %run scripts/vcp.py
vc_controller_version(vcp) 

VCCのバージョンが25.04以上であることをチェックします。

In [ ]:
 check_vcc_version(vcp, "25.04")

### group_varsの読み込み

次のセルを実行すると「010-パラメータの設定.ipynb」で指定したパラメータを読み込みます。読み込むパラメータの値は、UnitGroup名に指定した 値に対応するものになります。UnitGroup名の指定が誤っていると意図したパラメータが読み込めないので注意してください。

In [ ]:
%run scripts/group.py

gvars = load_group_vars(ugroup_name)

## mdx VMの準備

mdx上でOpenHPC環境を構築する場合に、マスターノードと計算ノードのVMをデプロイします。
mdx以外のクラウドを利用している場合は、この章をスキップして次章から実行してください。

### mdx操作の準備

mdx REST APIの認証トークンを入力し、IPv4接続を強制した上で、mdxプラグインを読み込みます。

In [ ]:
from getpass import getpass
mdx_token = getpass("mdx API token")

IPv4接続を強制するように設定します。

In [ ]:
%run scripts/mdx_ops.py
use_ipv4_only()

mdxプラグインを読み込みます。

In [ ]:
from vcpsdk.plugins.mdx_ext import MdxResourceExt
mdx = MdxResourceExt(mdx_token)
mdx.set_current_project_by_name(gvars['mdx_project_name'])

### VMスペックの生成

マスターノードと計算ノード（初期Feature）のVMスペックを、group_varsに保存されたパラメータから生成します。

In [ ]:
%run scripts/mdx_ops.py
import os

# SSH公開鍵の読み込み
with open(os.path.expanduser(gvars['ssh_public_key_path'])) as f:
    shared_key = f.read()

# マスターノード用VMスペック
mdx_master_spec = mdx_get_vm_spec(
    mdx,
    gvars['mdx_master_pack_num'],
    False,  # マスターノードはGPU不使用
    gvars['master_root_size'],
    gvars['mdx_segment_id'],
    shared_key
)

# 計算ノード用VMスペック（初期Feature）
partition_name = list(gvars["slurm_partitions"].keys())[0]
partition_config = gvars["slurm_partitions"][partition_name]
nodeset_name = partition_config["nodesets"][0]
feature_name = nodeset_name.replace("ns-", "")
feature_config = gvars["slurm_features"][feature_name]

mdx_compute_spec = mdx_get_compute_vm_spec_from_feature(
    mdx, feature_config, gvars['mdx_segment_id'], shared_key
)

# service_level の設定
from vcpsdk.plugins.mdx_ext import MDX_VM_SPEC_SCHEMA

if not 'service_level' in MDX_VM_SPEC_SCHEMA['properties']:
    MDX_VM_SPEC_SCHEMA['properties'].update(
        {'service_level': {'type': 'string'}}
    )

for s in [mdx_master_spec, mdx_compute_spec]:
    s.update({'service_level': 'guarantee'})

### VMデプロイ

マスターノードと計算ノードをmdx上にデプロイします。

In [ ]:
# マスターノードと計算ノードのホスト名リストを作成
hostname_template = feature_config["hostname_template"]
compute_hostnames = [
    hostname_template.format(i + 1)
    for i in range(feature_config["nodes"])
]
compute_ip_addresses = feature_config["ip_addresses"][:feature_config["nodes"]]

# マスターノードのデプロイ
mdx.deploy_vm(gvars['master_hostname'], mdx_master_spec.copy(), wait_for=False)
print(f"{gvars['master_hostname']} deployed.")

# 計算ノードのデプロイ
for hostname in compute_hostnames:
    mdx.deploy_vm(hostname, mdx_compute_spec.copy(), wait_for=False)
    print(f"{hostname} deployed.")

各VMの起動とIPアドレスの設定を待ち合わせます。5分程度かかります。

`Error: VMs not deployed: [vm1, vm2...]` のように表示された場合、VMのデプロイに失敗しています。

In [ ]:
from vcpsdk.plugins.mdx_ext import SLEEP_TIME_SEC, DEPLOY_VM_SLEEP_COUNT
import time
import re

mdx_vms = [gvars['master_hostname']] + compute_hostnames
vms_rest = mdx_vms.copy()

for i in range(DEPLOY_VM_SLEEP_COUNT):
    for vm in vms_rest.copy():
        info = mdx.get_vm_info(vm)
        if info["status"] != "PowerON":
            continue
        addr = info["service_networks"][0]["ipv4_address"][0]
        if re.match(r"^[0-9]+\.[0-9]+\.[0-9]+\.[0-9]+$", addr) is None:
            continue
        vms_rest.remove(vm)
        print(f"{vm} is now up with address {addr}.")
        break
    if len(vms_rest) == 0:
        break
    else:
        time.sleep(SLEEP_TIME_SEC)

if len(vms_rest) > 0:
    raise RuntimeError(f"Error: VMs not deployed: {vms_rest}")

### IPアドレスの変更

現時点のmdxでは、割り当てるIPアドレスを指定してVMをデプロイすることはできないため、起動したVMのIPアドレスを010で設定したアドレスに変更します。初回のログイン時にパスワード設定を求められるので、初期パスワードを設定した上でIPアドレスの変更を行います。

In [ ]:
import os

# マスターノード+計算ノードのIPアドレスマッピング
all_etc_hosts = {}
all_etc_hosts[gvars['master_ipaddress']] = gvars['master_hostname']
for addr, hostname in zip(compute_ip_addresses, compute_hostnames):
    all_etc_hosts[addr] = hostname

# 初期パスワード設定
initial_passwd = 'openhpcv3_mdx_vm_initial_password'
mdx_set_init_passwd(mdx, list(all_etc_hosts.values()),
                    gvars['ssh_private_key_path'],
                    initial_passwd)

# IPアドレス変更
mdx_change_ipaddrs(mdx, all_etc_hosts,
                   os.path.expanduser(gvars['ssh_private_key_path']),
                   verbose=True)

### VCP既存サーバモードのセットアップ

各VMを、VCP既存サーバ(SSH)モードでVCノードとして使用できるようにします。

In [ ]:
# VCノードとして使用できるように設定
vcppubkey = vcp.get_publickey()
mdx_init_vcp(list(all_etc_hosts.keys()),
             os.path.expanduser(gvars['ssh_private_key_path']),
             vcppubkey)

## VCディスクの作成

マスターノードのNFSサーバが公開するファイルを配置するためのディスクを作成します。

> 「010-パラメータ設定.ipynb」でVCディスクを作成しない設定にした場合は、この節を実行してもVCディスクは作成されません。

![VCディスク](images/ohpc-007.png)

### VCディスクのUnitGroup作成

VCディスクを管理するためのUnitGroupを作成します。

UnitGroupを作成する前に、現在のUnitGroupの一覧を確認します。

In [ ]:
vcp.df_ugroups()

VCディスクのUnitGroupを作成します。

In [ ]:
if 'nfs_disk_size' in gvars:
    ug_disk = vcp.create_ugroup(f'{ugroup_name}_disk', ugroup_type='storage')

UnitGroupの一覧を確認します。

In [ ]:
vcp.df_ugroups()

### VCディスクの作成

VCディスクを作成します。

In [ ]:
if 'nfs_disk_size' in gvars:
    provider = gvars['master_provider']
    spec_disk = vcp.get_spec(f'{provider}_disk', 'small')
    match gvars['master_provider']:
        case 'azure':
            spec_disk.disk_size_gb = gvars['nfs_disk_size']
        case 'oracle':
            spec_disk.size_in_gbs = gvars['nfs_disk_size']
        case _:
            spec_disk.size = gvars['nfs_disk_size']

    unit_disk = ug_disk.create_unit('nfs', spec_disk)

UnitGroupに作成したVCディスクの一覧を表示してみます。

In [ ]:
from IPython.display import display
if 'unit_disk' in vars() and unit_disk:
    display(unit_disk.df_nodes())

## VCノードのUnitGroup作成

VCノードを管理するためのUnitGroupを作成します。

UnitGroupを作成する前に、現在のUnitGroupの一覧を確認します。

In [ ]:
vcp.df_ugroups()

VCノードのUnitGroupを作成します。

In [ ]:
ug = vcp.create_ugroup(ugroup_name)

UnitGroupの一覧を確認します。

In [ ]:
vcp.df_ugroups()

## マスターノードのセットアップ

マスターノードとして利用するVCノードのセットアップを行います。

![マスターノード](images/ohpc-008.png)

### VCノードの起動

マスターノードとして利用するVCノードを起動します。

#### spec を指定する

「010-パラメータ設定.ipynb」で指定したパラメータを`spec` に設定します。

`spec`に設定するパラメータを以下に示します。

* `image`: Baseコンテナイメージ
  - Baseコンテナイメージを設定します
* `params_v`: ボリューム設定
  - Baseコンテナのボリュームを設定します
  - OpenHPCコンテナではホスト側の `/sys/fs/cgroup`をコンテナから見えるように設定する必要があります
* `ip_addresses`: IPアドレス
  - VCノードに割り当てるプライベートIPアドレスを設定します
* `disks`: VCディスクのリスト
  - VCノードにアタッチするVCディスクのリストを設定します
* `volume_size`: ルートボリュームサイズ
  - VCノードのルートボリュームサイズを設定します
* `instance_type`: インスタンスタイプ
  - VCノードのインスタンスタイプ
* `set_ssh_publickey()`: SSHの公開鍵
  - VCノードに登録するSSHの公開鍵

In [ ]:
spec_master = vcp.get_spec(gvars['master_provider'], gvars['master_flavor'])
spec_master.image = 'harbor.vcloud.nii.ac.jp/vcp/openhpc:master-3.4-multi'
spec_master.params_v = [
    '/sys/fs/cgroup:/sys/fs/cgroup:rw',
    '/lib/modules:/lib/modules:ro',
]
# onpremisesの場合にはnum_nodesを明示的に指定
if gvars['master_provider'] == 'onpremises':
    spec_master.num_nodes = 1
spec_master.ip_addresses = [gvars['master_ipaddress']]
if 'ug_disk' in vars() and ug_disk:
    spec_master.disks = ug_disk.find_nodes()
else:
    spec_master.params_v.append('/srv/OpenHPC:/exports')

match gvars['master_provider']:
    case 'aws':
        spec_master.volume_size = gvars['master_root_size']
        if 'master_instance_type' in gvars:
            spec_master.instance_type = gvars['master_instance_type']
    case 'azure':
        spec_master.disk_size_gb = gvars['master_root_size']
        if 'master_instance_type' in gvars:
            spec_master.vm_size = gvars['master_instance_type']
    case 'oracle':
        spec_master.boot_volume_size_in_gbs = gvars['master_root_size']
        if 'master_instance_type' in gvars:
            spec_master.shape = gvars['master_instance_type']

spec_master.set_ssh_pubkey(gvars['ssh_public_key_path'])

spec_master.hostname = gvars["master_hostname"]
spec_master.dns = [gvars["master_ipaddress"]]
spec_master.dns_search = [gvars["dns_domain"]]

mdxの場合には、VCノードとなるVMにログインするためのユーザ名を指定します。

In [ ]:
if 'mdx_ssh_user_name' in gvars:
    spec_master.user_name = gvars['mdx_ssh_user_name']

ここまで `spec` に設定した内容を表示してみます。

In [ ]:
print(spec_master)

ベースコンテナの環境変数に以下のパラメータを設定します。

* `MUNGE_KEY`
    - VCノードの `munge.key` の内容
* `NFS_DEV`
    - NFS用ディスクのデバイス名
* `CLUSTER`
    - Slurmのクラスタ名
    - VCPのUnitGroup名と同じ値になる
* `ENABLE_NSS_SLURM`
    - [Name Service Caching Through NSS Slurm](https://slurm.schedmd.com/archive/slurm-23.11.10/nss_slurm.html)を有効にする

In [ ]:
%run scripts/utils.py

munge_key = spec_env_munge_key(gvars, vcp, vcc_access_token, verify=gvars.get("verify_ssl_certificate", True))
spec_master.params_e.extend([
    f'MUNGE_KEY={munge_key}',
    f'CLUSTER={gvars["ugroup_name"]}',
    'ENABLE_NSS_SLURM=1',
])

if 'nfs_device' in gvars:
    spec_master.params_e.append(f'NFS_DEV={gvars["nfs_device"]}')

`spec` に設定した内容を表示してみます。

> 表示内容には、**秘匿情報**となる `munge.key` の内容を含んでいます。扱いには**注意してください**。

In [ ]:
print(spec_master)

#### VCノードの起動

マスターノードとして利用するVCノードを起動します。

> AWSで起動するにはおよそ15分程度かかります。

In [ ]:
unit_master = ug.create_unit('master', spec_master)

VCノードの状態が `RUNNING` になっていることを確認します。

> VCノードの起動に失敗して`RUNNING`以外の状態になっている場合は次のセルを実行するとエラーになります。

In [ ]:
unit_master = ug.get_unit('master')
if any([node.state != 'RUNNING' for node in unit_master.find_nodes()]):
    raise RuntimeError('ERROR: not running')

起動したVCノードの一覧を表示します。

> 上のセルがエラーになった場合も、次のセルを実行してノードの状態を確認してください。`node_state` が`BOOTING`だった場合は、単にノード起動に時間がかかっているだけである可能性があります。しばらく待ってから、ノードの状態を再確認してみてください。また、`node_state`が`ERROR`となっている場合はノードの起動に失敗しています。`ug.cleanup()` を実行して VCノードを削除してください。その後、エラー原因を解消してから「[VCノードのUnitGroup作成](#VCノードのUnitGroup作成)」以下を `unfreeze` して再実行してください。

In [ ]:
unit_master.df_nodes()

#### NFS serverの起動確認

NFS serverのサービスが実行されていることを確認します。

In [ ]:
addr = gvars['master_ipaddress']
ssh_opts = f'-o UserKnownHostsFile=/dev/null -o StrictHostKeyChecking=no -i {gvars["ssh_private_key_path"]}'
!ssh-keygen -R {addr}
!ssh {ssh_opts} vcp@{addr} systemctl status nfs-server

エラーになった場合は、起動したVCノードにて `journalctl` などを実行してエラーの原因を確認してください。原因となった事象を解消した後に次のセルのコメントを外してから実行し NFSサーバのサービスを再起動してください。

> よくある事例として、起動に時間がかかりタイムアウトすることでNFSサーバが起動しないことがあります。この場合は、単に次のセルのコメントを外してサービスの再起動を行うことで問題が解消します。

In [ ]:
# !ssh {ssh_opts} vcp@{addr} sudo systemctl restart nfs-server

サービスを再起動した場合は次のセルのコメントを外して、NFSサーバのサービスが起動したことを確認してください。

In [ ]:
# !ssh {ssh_opts} vcp@{addr} systemctl status nfs-server

NFSのエクスポート設定が１つ以上あることを確認します。次のセルを実行し、エラーとならないことを確認してください。

In [ ]:
# エクスポートの設定状況を表示する
!ssh {ssh_opts} vcp@{addr} "sudo /sbin/exportfs -v"
# エクスポートの設定項目が１つ以上あることをチェックする
!test $(ssh {ssh_opts} vcp@{addr} "sudo /sbin/exportfs -v" | wc -l) -gt 0 

上のセルがエラーとなる場合は、パラメータ設定にて指定した `nfs_device` の値が正しくないことが考えられます。次のセルのコメントを外してパラメータに設定した値と、起動したVCノードにアタッチされているブロックデバイス名が一致しているかを確認してください。正しく設定されている場合は、以下の表示例のように１行目と最後の行に表示されるブロックデバイス名（この例では`xvdf`） に同じ値が表示されます。

```
nfsdevice=/dev/xvdf
NAME    MAJ:MIN RM SIZE RO TYPE MOUNTPOINT
xvda    202:0    0  60G  0 disk 
└─xvda1 202:1    0  60G  0 part /etc/hosts
xvdf    202:80   0  64G  0 disk /exports
```

異なる値が表示されている場合は設定するパラメータを修正する必要があります。
「920-OpenHPC環境の削除.ipynb」のNotebookで起動したVCノード、VCディスクを削除してから、再度「010-パラメータ設定.ipynb」のNotebookで、正しいブロックデバイス名を設定しなおしてください。

In [ ]:
# 設定されているパラメータ nfs_device の値を確認する
# print(f'nfsdevice={gvars["nfs_device"]}')

# VCノードのブロックデバイスを確認する
# !ssh {ssh_opts} vcp@{addr} lsblk 2> /dev/null

### Ansibleのセットアップ

マスターノードに対してAnsibleで操作を行うための設定を行います。

まず、VCノードにSSHでログインできるようにするために `~/.ssh/known_hosts` の更新を行います。

> 何度かVCノードの起動を行うと、異なるホストが同じIPアドレスで起動するためにSSHのホストキーのチェックでエラーになる事があります。このような状況に対応するために、起動したVCノードのIPアドレスに対応するエントリを`known_hosts`ファイルから削除します。その後、`ssh-keyscan`コマンドを利用して起動したVCノードのホストキーを取得して `known_hosts`ファイルの内容を更新します。

In [ ]:
from time import sleep

def check_update_known_hosts(ipaddr):
    # VCノード起動直後だと sshd サービスが開始されておらずに known_hosts が更新されない場合がある
    # ssh-keyscan が値を取得できるまで何度かリトライする
    for x in range(10):
        out = ! echo $(ssh-keyscan {ipaddr} 2> /dev/null | wc -l)
        update_lines = int(out[0])
        if update_lines > 0:
            break
        sleep(1)
    else:
        raise RuntimeError("ERROR: timeout!")    

!mkdir -p -m 0700 ~/.ssh
!touch ~/.ssh/known_hosts
for addr in unit_master.find_ip_addresses():
    !ssh-keygen -R {addr}
    check_update_known_hosts(addr)
    !ssh-keyscan -H {addr} >> ~/.ssh/known_hosts

起動したVCノードに対応するエントリを Ansible のインベントリに登録します。

> Ansibleで操作を行うためには、操作対象のホスト(IPアドレス)をインベントリに登録する必要があります。

In [ ]:
%run scripts/group.py

inventory = {
    'all': {
        'children': {
            ugroup_name: {
                'children': {
                    f'{ugroup_name}_master': {
                        'hosts': {ip: {} for ip in unit_master.find_ip_addresses()},
                    },
                },
                'vars': {
                    'ansible_user': 'vcp',
                    'ansible_ssh_private_key_file': gvars['ssh_private_key_path'],
                },
            }
        }
    }
}

update_inventory_yml(inventory)

!cat inventory.yml

先程VCノードを登録したファイルをインベントリとして指定するためのAnsibleのコンフィギュレーションファイルを作成します。

> カレントディレクトリにコンフィギュレーションファイル(`ansible.cfg`)を作成すると、Ansibleを実行する際にその設定が適用されます。

In [ ]:
from pathlib import Path

inventory_path = Path('./inventory.yml').resolve()
ansible_cfg = Path('./ansible.cfg').resolve()
with ansible_cfg.open(mode='w') as f:
    f.write(f'''
[defaults]
inventory = {inventory_path}
remote_tmp = /var/tmp
timeout = 30
force_valid_group_names = ignore
''')
ansible_cfg.parent.chmod(0o755)

!cat ./ansible.cfg

VCノードに対して Ansible で接続できることを確認します。

In [ ]:
!ansible {ugroup_name}_master -m ping

正常に接続できると以下のように表示されます。

```
XXX.XXX.XXX.XXX | SUCCESS => {
    "changed": false, 
    "ping": "pong"
}
```

VCノードに対して設定ファイルの変更やパッケージの追加を行う場合にVCノードの管理者権限が必要になる場合があります。Ansibleで管理者権限によるコマンド実行が可能かどうかを確認します。

In [ ]:
!ansible {ugroup_name}_master -b -a 'whoami'

### Slurm設定のセットアップ

マスターノードで `slurm.conf` と `hosts.ohpc` を生成します。

**重要**: この手順は計算ノードを起動する前に完了させる必要があります。
計算ノードは起動時にCoreDNSでホスト名を解決し、Slurmに自動登録されます。
そのため事前に`hosts.ohpc`が作成され、また`slurm.conf`にパーティション定義が必要になります。

Ansibleを使用してマスターノードに以下のファイルを生成します：

- **slurm.conf**: Slurmの設定ファイル
  - Feature/Nodeset/Partition定義
  - MaxNodeCount（全Featureの合計ノード数）
  - GPU使用時はGresTypes設定
- **hosts.ohpc**: CoreDNS用のhostsファイル
  - 全計算ノードのホスト名とIPアドレスのマッピング

まずチェックモードで実行します。

In [ ]:
!ansible-playbook -l {ugroup_name}_master -v -CD playbooks/setup-slurm.yml

実際に変更を行います。

In [ ]:
!ansible-playbook -l {ugroup_name}_master -v playbooks/setup-slurm.yml

生成された設定を確認します：

1. **slurm.conf**: Slurm設定ファイルの内容確認
   - MaxNodeCount: 全Featureのノード数合計が正しいか
   - GresTypes: GPU使用時に設定されているか
   - NodeSet: `ns-{feature_name}` 形式で定義されているか
   - PartitionName: パーティション定義が正しいか

2. **hosts.ohpc**: CoreDNS用hostsファイルの内容確認
   - 全計算ノードのエントリが記載されているか
   - IPアドレスとホスト名のマッピングが正しいか

3. **CoreDNS**: サービスの稼働状態確認
   - サービスが正常に起動しているか

In [ ]:
# 1. slurm.conf の確認
print("=== Slurm Configuration ===")
!ansible {ugroup_name}_master \
    -m shell -a "scontrol show config | grep -E 'MaxNodeCount|GresTypes'"

# 2. hosts.ohpc の確認
print("\n=== Hosts File (hosts.ohpc) ===")
!ansible {ugroup_name}_master \
    -m shell -a "cat /opt/ohpc/pub/etc/hosts.ohpc"

# 3. CoreDNS の確認
print("\n=== CoreDNS Service Status ===")
!ansible {ugroup_name}_master \
    -a "systemctl is-active coredns"

## 計算ノードのセットアップ


計算ノードとして利用するVCノードのセットアップを行います。

![計算ノード](images/ohpc-009.png)

### VCノードの起動

計算ノードとして利用するVCノードを起動します。

#### spec を指定する

「010-パラメータ設定.ipynb」で指定したパラメータを`spec` に設定します。

`spec`に設定するパラメータを以下に示します。

* `image`: Baseコンテナイメージ
  - Baseコンテナイメージを設定します
* `params_v`: ボリューム設定
  - Baseコンテナのボリュームを設定します
  - OpenHPCコンテナではホスト側の `/sys/fs/cgroup`をコンテナから見えるように設定する必要があります
* `ip_addresses`: IPアドレス
  - VCノードに割り当てるプライベートIPアドレスを設定します
* `volume_size`: ルートボリュームサイズ
  - VCノードのルートボリュームサイズを設定します
* `instance_type`: インスタンスタイプ
  - VCノードのインスタンスタイプ
* `set_ssh_publickey()`: SSHの公開鍵
  - VCノードに登録するSSHの公開鍵

In [ ]:
# 初期パーティションから最初のFeatureを取得
partition_name = list(gvars["slurm_partitions"].keys())[0]
partition_config = gvars["slurm_partitions"][partition_name]
nodeset_name = partition_config["nodesets"][0]  # 初期構築時は1つのみ

# Nodeset名からFeature名を抽出（"ns-{feature_name}" 形式）
feature_name = nodeset_name.replace("ns-", "")

# Feature設定を取得
feature_config = gvars["slurm_features"][feature_name]

# VCP spec の設定
spec_compute = vcp.get_spec(feature_config['provider'], feature_config.get('flavor'))

# GPU設定
if feature_config['use_gpu']:
    spec_compute.gpus = 'all'
    spec_compute.image = 'harbor.vcloud.nii.ac.jp/vcp/openhpc:compute-gpu-3.4-multi'
    if feature_config['provider'] != 'onpremises':
        spec_compute.cloud_image = 'niivcp-25.04.0-x86_64-ubuntu24.04-gpu-a-r1.0'
else:
    spec_compute.image = 'harbor.vcloud.nii.ac.jp/vcp/openhpc:compute-3.4-multi'

# bind mount設定
spec_compute.params_v = [
    '/sys/fs/cgroup:/sys/fs/cgroup:rw',
    '/opt/var/lib/docker:/var/lib/docker',
    '/lib/modules:/lib/modules:ro',
]

# onpremisesの場合にはnum_nodesを明示的に指定
if feature_config['provider'] == 'onpremises':
    spec_compute.num_nodes = feature_config['nodes']

# IPアドレスの設定
spec_compute.ip_addresses = feature_config['ip_addresses'][:feature_config['nodes']]

# インスタンスタイプとルートサイズ（オプション）
if 'root_size' in feature_config:
    match feature_config['provider']:
        case 'aws':
            spec_compute.volume_size = feature_config['root_size']
        case 'azure':
            spec_compute.disk_size_gb = feature_config['root_size']
        case 'oracle':
            spec_compute.boot_volume_size_in_gbs = feature_config['root_size']

if 'instance_type' in feature_config:
    match feature_config['provider']:
        case 'aws':
            spec_compute.instance_type = feature_config['instance_type']
        case 'azure':
            spec_compute.vm_size = feature_config['instance_type']
        case 'oracle':
            spec_compute.shape = feature_config['instance_type']

# ssh公開鍵
spec_compute.set_ssh_pubkey(gvars['ssh_public_key_path'])

# DNS設定
spec_compute.dns = [gvars["master_ipaddress"]]
spec_compute.dns_search = [gvars["dns_domain"]]
spec_compute.add_host = [f"master:{gvars['master_ipaddress']}"]

mdxの場合には、VCノードとなるVMにログインするためのユーザ名を指定します。

In [ ]:
if 'mdx_ssh_user_name' in gvars:
    spec_compute.user_name = gvars['mdx_ssh_user_name']

ここまで `spec` に設定した内容を表示してみます。

In [ ]:
print(spec_compute)

ベースコンテナの環境変数に以下のパラメータを設定します。

* `MUNGE_KEY`
    - VCノードの `munge.key` の内容
* `MASTER_HOSTNAME`
    - マスターノードのホスト名
* `CLUSTER`
    - Slurmのクラスタ名
    - VCPのUnitGroup名と同じ値になる
* `FEATURE`
    - ノードのFeatureを指定する
* `ENABLE_NSS_SLURM`
    - [Name Service Caching Through NSS Slurm](https://slurm.schedmd.com/archive/slurm-23.11.10/nss_slurm.html)を有効にする
* `GPUS`
    - 計算ノードのGPU数を指定する

In [ ]:
%run scripts/utils.py

munge_key = spec_env_munge_key(gvars, vcp, vcc_access_token, verify=gvars.get("verify_ssl_certificate", True))
spec_compute.params_e.extend([
    f'MUNGE_KEY={munge_key}',
    f'MASTER_HOSTNAME={gvars["master_hostname"]}',
    f'CLUSTER={gvars["ugroup_name"]}',
    f'FEATURE={feature_name}',
    'ENABLE_NSS_SLURM=1',
])

# GPU数の設定
if feature_config['use_gpu'] and feature_config.get('gpus', 0) > 0:
    spec_compute.params_e.append(f'GPUS={feature_config["gpus"]}')

# オプショナルNFSエントリ
if feature_config.get('nfs_entries'):
    # nfs_entriesは既にBase64エンコード済み
    spec_compute.params_e.append(f"OPTIONAL_NFS_FSTAB_BASE64={feature_config['nfs_entries']}")

`spec` に設定した内容を表示してみます。

> 表示内容には、**秘匿情報**となる `munge.key` の内容を含んでいます。扱いには**注意してください**。

In [ ]:
print(spec_compute)

#### VCノードの起動

計算ノードとして利用するVCノードを起動します。

VCユニット名を確認します。 VCユニット名は、計算ノードのFeature名と同じ値になります。

In [ ]:
vc_unit_name = feature_config.get('vc_unit', feature_name)
vc_unit_name

実際にVCノードを起動します。

> AWSで起動するにはおよそ３分程度かかります。

In [ ]:
unit_compute = ug.create_unit(vc_unit_name, spec_compute)

VCノードの状態が `RUNNING` になっていることを確認します。

> VCノードの起動に失敗して`RUNNING`以外の状態になっている場合は次のセルを実行するとエラーになります。エラーになった場合は、`ug.cleanup()` を実行して VCノードを削除してください。その後、「[VCノードのUnitGroup作成](#VCノードのUnitGroup作成)」以下を `unfreeze` してから再実行してください。

In [ ]:
unit_compute = ug.get_unit(vc_unit_name)
if any([node.state != 'RUNNING' for node in unit_compute.find_nodes()]):
    raise RuntimeError('ERROR: not running')

起動したVCノードの一覧を表示します。

In [ ]:
unit_compute.df_nodes()

### Ansibleのセットアップ

計算ノードに対してAnsibleで操作を行うための設定を行います。

まず`~/.ssh/known_hosts` の更新を行います。

> 何度かVCノードの起動を行うと、異なるホストが同じIPアドレスで起動するためにSSHのホストキーのチェックでエラーになる事があります。このような状況に対応するために、起動したVCノードのIPアドレスに対応するエントリを`known_hosts`ファイルから削除します。その後、`ssh-keyscan`コマンドを利用して起動したVCノードのホストキーを取得して `known_hosts`ファイルの内容を更新します。

In [ ]:
for addr in unit_compute.find_ip_addresses():
    !ssh-keygen -R {addr}
    check_update_known_hosts(addr)
    !ssh-keyscan -H {addr} >> ~/.ssh/known_hosts

起動した計算ノードに対応するエントリを Ansible のインベントリに登録します。

> Ansibleで操作を行うためには、操作対象のホスト(IPアドレス)をインベントリに登録する必要があります。

In [ ]:
%run scripts/group.py
import yaml

with open('inventory.yml', 'r') as f:
    inventory = yaml.safe_load(f)

# 計算ノードを追加
compute_group_name = f'{ugroup_name}_{vc_unit_name}'
inventory['all']['children'][ugroup_name]['children'][compute_group_name] = {
    'hosts': {ip: {} for ip in unit_compute.find_ip_addresses()},
}

# inventory を更新
update_inventory_yml(inventory)

# inventoryのノード構成を確認
!ansible-inventory --graph {ugroup_name}

VCノードに対して Ansible で接続できることを確認します。

In [ ]:
!ansible {ugroup_name} -m ping

正常に接続できると以下のように表示されます。

```
XXX.XXX.XXX.XXX | SUCCESS => {
    "changed": false, 
    "ping": "pong"
}
```

VCノードに対して設定ファイルの変更やパッケージの追加を行う場合にVCノードの管理者権限が必要になる場合があります。Ansibleで管理者権限によるコマンド実行が可能かどうかを確認します。

In [ ]:
!ansible {ugroup_name} -b -a 'whoami'

## Slurmの状態を確認する

起動したVCノードで実行されているSlurmの状態を確認します。

### ノードの状態を確認する

マスターノード起動時点では計算ノードがまだNodesetに参加していないため、Slurmはそのまま起動していてもPartitionを認識しない場合があります。計算ノード起動後に scontrol reconfigure を実行してSlurmの状態を更新します。

In [ ]:
!ansible {ugroup_name}_master -b -a "scontrol reconfigure"

Slurmクラスタのノードの状態を確認します。

In [ ]:
!ansible {ugroup_name}_master -a sinfo

計算ノードが２ノードの場合、以下のような実行結果が表示されます。
```
PARTITION AVAIL  TIMELIMIT  NODES  STATE NODELIST
compute*     up   infinite      2   idle c[1-2]
```

> **アクセス制御を設定している場合**: `partition_allow_groups` を指定して構築した場合、
> Slurmの `AllowGroups` パラメータが設定されています。この場合、指定したグループに所属していない
> ユーザはそのPartitionにジョブを投入できません。次節で設定内容を確認してください。

ノードごとの詳細な状態を表示します。

In [ ]:
!ansible {ugroup_name}_master -a "scontrol show node"

### Partitionのアクセス制御を確認する（オプション）

`partition_allow_groups` を設定して構築した場合は、Partitionに `AllowGroups` が設定されています。
この節ではアクセス制御の設定内容とグループメンバーシップを確認します。

> アクセス制御を設定していない場合は、この節をスキップしてください。

In [ ]:
# Partitionの AllowGroups 設定を確認する
!ansible {ugroup_name}_master -m shell \
    -a "scontrol show partition | grep -E 'PartitionName|AllowGroups'"

マスターノード上のユーザ(vcp)のグループ所属を確認します。

In [ ]:
check_user = "vcp"
!ansible {ugroup_name}_master -m shell -a "id {check_user}"

**アクセス制御が設定されている場合の注意事項**:

- `AllowGroups` に指定したグループに所属していないユーザがジョブを投入しようとすると、以下のようなエラーになります：
```
sbatch: error: Batch job submission failed: User's group not permitted to use this partition
```
- ユーザをグループに追加するには、マスターノードおよび全計算ノードで以下を実行します：
```bash
sudo usermod -aG <グループ名> <ユーザ名>
```
- グループへの追加後、ユーザは**再ログイン**してグループの変更を反映させる必要があります。

### ジョブを実行する

ジョブを実行できることを確認します。

> **アクセス制御を設定している場合**: `srun` や `sbatch` を実行するユーザが、対象Partitionの
> `AllowGroups` に指定したグループに所属している必要があります。
> 所属していない場合は `User's group not permitted to use this partition` エラーが発生します。

`srun`コマンドで、`hostname` コマンドを計算ノードで実行させてみます。

計算ノードが２ノードの場合、以下のような実行結果が表示されます（ホストの表示順序は入れ替わることがあります）。

```
0: c1
1: c2
```

In [ ]:
compute_nodes = len(unit_compute.find_ip_addresses())
!ansible {ugroup_name}_master -a 'srun -l -N {compute_nodes} hostname'

次に、サンプルの `hello.c` をコンパイルして `sbatch` で実行してみます。

まず、ソースファイルをコンパイルして、実行ファイルを作成します。

In [ ]:
!ansible {ugroup_name}_master -a 'mpicc -O3 -o hello /opt/ohpc/pub/examples/mpi/hello.c'
!ansible {ugroup_name}_master -a 'ls -l hello'

ジョブスクリプトを作成します。

In [ ]:
from tempfile import TemporaryDirectory
from pathlib import Path

with TemporaryDirectory() as work_dir:
    job_file = Path(work_dir) / 'hello.job'
    with job_file.open(mode='w') as f:
        f.write(f'''#!/bin/bash

#SBATCH -J hello               # Job name
#SBATCH -o hello.%j.out        # Name of stdout output file (%j expands to jobId)
#SBATCH -N {compute_nodes}     # Total number of nodes requested
#SBATCH -n {compute_nodes}     # Total number of mpi tasks requested
#SBATCH -t 00:01:00            # Run time (hh:mm:ss)

# Launch MPI-based executable
prun ./hello
''')
    !cat {job_file}
    !ansible {ugroup_name}_master -m copy -a 'src={str(job_file)} dest=.'

ジョブを投入します。

In [ ]:
!ansible {ugroup_name}_master -a 'sbatch hello.job'

ジョブの出力結果を表示してみます。ジョブの実行が成功していると以下のような出力結果が得られます。

```
[prun] Master compute host = c1
[prun] Resource manager = slurm
[prun] Launch cmd = mpirun ./hello (family=openmpi4)

 Hello, world (2 procs total)
    --> Process #   0 of   2 is alive. -> c1
    --> Process #   1 of   2 is alive. -> c2
```

In [ ]:
!ansible {ugroup_name}_master -m shell -a 'cat hello.*.out'